# DEH 30-Day PySpark Challenge
## Day 21 — Mini Project: End-to-End Pipeline

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. Run each cell in order using `Shift + Enter`
3. Build the pipeline in the working cells below — try it yourself before checking the reference solution at the bottom

---

## The Brief

Build a **customer revenue summary** report:
- One row per customer
- Total spend, order count, average order value
- Their single most expensive order
- Customer segment and region
- Written as Parquet, partitioned by region

See the Day 21 lesson page for the full brief and suggested pipeline steps.

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

## Step 1 — Read all source files with explicit schemas

In [ ]:
orders_schema = StructType([
    StructField("order_id",       StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("order_date",     DateType(),    True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("discount_pct",   IntegerType(), True),
    StructField("status",         StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("region",         StringType(),  True)
])

customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name",  StringType(), True),
    StructField("last_name",   StringType(), True),
    StructField("email",       StringType(), True),
    StructField("city",        StringType(), True),
    StructField("state",       StringType(), True),
    StructField("country",     StringType(), True),
    StructField("signup_date", DateType(),   True),
    StructField("segment",     StringType(), True)
])

products_schema = StructType([
    StructField("product_id",     StringType(), True),
    StructField("product_name",   StringType(), True),
    StructField("category",       StringType(), True),
    StructField("sub_category",   StringType(), True),
    StructField("unit_price",     DoubleType(), True),
    StructField("cost_price",     DoubleType(), True),
    StructField("supplier",       StringType(), True),
    StructField("stock_quantity", IntegerType(),True)
])

orders_df    = spark.read.option("header","true").schema(orders_schema).csv(f"s3a://{BUCKET}/data/orders.csv")
customers_df = spark.read.option("header","true").schema(customers_schema).csv(f"s3a://{BUCKET}/data/customers.csv")
products_df  = spark.read.option("header","true").schema(products_schema).csv(f"s3a://{BUCKET}/data/products.csv")

print(f"Orders: {orders_df.count()} | Customers: {customers_df.count()} | Products: {products_df.count()}")

## Try it yourself

Use the empty cells below to build your pipeline. Steps 2-8 are outlined on the lesson page. Don't scroll to the reference solution until you've made a real attempt.

In [ ]:
# Step 2 — Clean the data (drop duplicate orders)


In [ ]:
# Step 3 — Join orders with customers


In [ ]:
# Step 4 — Compute revenue per order


In [ ]:
# Step 5 — Group by customer and aggregate


In [ ]:
# Step 6 — Find each customer's most expensive order using a window function


In [ ]:
# Step 7 — Join aggregated summary with top order and customer details


In [ ]:
# Step 8 — Write final result as Parquet partitioned by region, then verify


## Deliverable 3 — Answer in this cell

Which customer segment has the highest average order value?  
Which region has the most customers in the top-spender tier (top 5 by total_spend)?

*(Write your answer here after running the analysis above)*

In [ ]:
# Your analysis code for Deliverable 3


---
## Reference Solution

Only check this after attempting the pipeline yourself. There are many valid ways to build this — yours doesn't need to match exactly.

In [ ]:
# Step 2 — Clean: drop duplicate orders by order_id
orders_clean = orders_df.dropDuplicates(["order_id"])

# Step 3 — Join orders with customers (left join — keep all orders)
orders_with_customer = orders_clean.join(customers_df, on="customer_id", how="left")

# Step 4 — Compute revenue per order
orders_with_revenue = orders_with_customer.withColumn(
    "revenue",
    F.round(F.col("unit_price") * F.col("quantity") * (1 - F.col("discount_pct") / 100), 2)
)

# Step 5 — Group by customer and aggregate
customer_summary = orders_with_revenue.groupBy(
    "customer_id", "first_name", "last_name", "segment", "region"
).agg(
    F.count("order_id").alias("order_count"),
    F.round(F.sum("revenue"), 2).alias("total_spend"),
    F.round(F.avg("revenue"), 2).alias("avg_order_value")
)

# Step 6 — Find each customer's top order using a window function
top_order_window = Window.partitionBy("customer_id").orderBy(F.col("revenue").desc())

top_orders = orders_with_revenue.withColumn(
    "row_num", F.row_number().over(top_order_window)
).filter(F.col("row_num") == 1).select(
    F.col("customer_id"),
    F.col("order_id").alias("top_order_id"),
    F.col("revenue").alias("top_order_revenue")
)

# Step 7 — Join the aggregated summary with the top order
final_summary = customer_summary.join(top_orders, on="customer_id", how="left")

final_summary.orderBy(F.col("total_spend").desc()).show(10)

In [ ]:
# Step 8 — Write to S3 as Parquet partitioned by region
final_summary.write \
    .mode("overwrite") \
    .partitionBy("region") \
    .parquet(f"s3a://{BUCKET}/output/customer_revenue_summary/")

# Verify
verify_df = spark.read.parquet(f"s3a://{BUCKET}/output/customer_revenue_summary/")
print(f"Rows written: {final_summary.count()}")
print(f"Rows read back: {verify_df.count()}")
verify_df.show(5)

In [ ]:
# Deliverable 3 — Reference analysis

# Segment with highest average order value
print("Average order value by segment:")
final_summary.groupBy("segment").agg(
    F.round(F.avg("avg_order_value"), 2).alias("segment_avg_order_value")
).orderBy(F.col("segment_avg_order_value").desc()).show()

# Top 5 spenders and their regions
print("Top 5 spenders by region:")
final_summary.orderBy(F.col("total_spend").desc()).limit(5) \
    .groupBy("region").count() \
    .orderBy(F.col("count").desc()) \
    .show()

---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*